# Addendum: Transformer Based Text Classification
Transformer models are pre-trained "machine learning" models you can download to do specific tasks. Sentiment classification models are common in this field. 

Machine learning models are trained by humans manually classifiying examples into different categories. The models are then shown some of these documents through a training process, and then the remainder are used to as tests, where the model is shown examples it's never seen before and asked how it would classify it. The more it matches the human classification, the better the model is considered to be.

You can get machine learning models for lots of different types of tasks. Large language models like ChatGPT and Google Gemini are evolutions of this kind of modelling.

Below we use the `transformers` library to download and set up the pre-trained model so we can pass it texts for classification. `transformers` is a Python library created by ['Hugging Face'](https://huggingface.co/models) a company that hosts and shares trained AI models.



In [ ]:
from transformers import pipeline


# The defaul 'sentiment-analysis' pipeline uses a model that just classifies into positive or negative
# get_sentiment = pipeline("sentiment-analysis")

# Other models classify differently. This model for example will also classify as neutral.
get_sentiment = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Repeating here just for reference
sentences = ["That sounds good.", # Positive
             "I love my new record player", # More Positive
               "I really hate it when my brother steals my things", # Negative
                 "I am a human"] # Neutral

get_sentiment(sentences)

These models work differently to Vader. 
- They can only `label`, rather than give a range or degree of sentiment.
- The `score` is not degree of sentiment, but how confident the model is in the label it has given.
- Generally the labels are correct except the default model assigns 'Positive' to our neutral statement, because it was only trained on recognising positive and negative. It would be better understood as labelling things as either negative, or not.

If we try the other model we'll see that whilst it assigns neutral to the final sentence, it also assigns it to the second one we'd consider more positive. It's not clear whether either model is 'better', nor whether our own classification of 'positive' is even correct. Confusing!

Things do not improve when we test the `confusing_sentences`

In [ ]:
confusing_sentences = ["the party was sick",
                        "She's got such a great mind. She's savage",
                          "Awesome, another parking ticket! Just what I need!",
                          "I absolutely love your ugly Christmas sweater! It is so ugly!"]

get_sentiment(confusing_sentences)

They also have a length limit that they can only understand documents of a maximum length, well below a typical news article. Generally they are trained on sentences rather than full pieces of text. This means they're a bit tricky to work with for anything other than short documents.

In [ ]:
import pandas as pd

articles = pd.read_parquet('farright_dataset_cleaned.parquet')
single_article = articles.loc[0,'cleaned_text']
# Running on the whole article will get us an error about the 'size of the tensor', essentially the document is too big.
get_sentiment(single_article)


It's possible to apply it by using spacy to break the document into sentences first. Then we treat each single article as a list of sentence length documents and get a label for each sentence. Then the question is how do you report that. You can turn them into numbers (-1,0,1) and take the average, but that tends to drift towards neutral. You could report on the counts of each of the three classifications but with news articles you will tend towards neutral as most sentences are conveying information, it is only occasionally that a single sentence will convey a stronger sentiment.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

article_sents = [sent.text for sent in nlp(single_article).sents]
sentence_sentiments = get_sentiment(article_sents)

scoring_dict = {'neutral':0, 'positive':1, 'negative':-1}

sentiment_numbers = pd.Series([scoring_dict[record['label']] for record in sentence_sentiments])

print(sentiment_numbers.mean())
print(sentiment_numbers.value_counts())